# Enclave Inference — Gemma 3 — Model Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `ENCLAVE_EMAIL` | Trusted execution environment |
| **Model owner** | (this notebook) | Owns the Gemma 3 model (weights + inference engine) |
| **Benchmark owner** | (separate notebook) | Owns AI safety evaluation prompts |
| **Researcher** | `RESEARCHER_EMAIL` | Submits inference job for bias/safety evaluation |

This notebook drives only the Model Owner steps; the Benchmark Owner and Researcher run their own notebooks in parallel.

The inference engine is private too, so before the evaluation the Model Owner has the enclave run [`syft-restrict`](https://github.com/OpenMined/PySyft/tree/dev/packages/syft-restrict) over it — proving it only does allow-listed JAX math, without revealing the architecture.

## Setup

In [ ]:
!uv pip install -Uq "jax[cpu]" flax orbax-checkpoint sentencepiece kagglehub==1.0.2 "git+https://github.com/OpenMined/PySyft.git@dev#subdirectory=packages/syft-enclave"

In [ ]:
!wget -nc https://raw.githubusercontent.com/OpenMined/PySyft/refs/heads/dev/notebooks/enclave/gemma/colab/gemma_inference_restrict.py

In [ ]:
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds
from gemma_inference_restrict import MODEL_CONFIGS

# ─── Choose model size here ─────────────────────────────────────────────────
MODEL_SIZE = "270m"  # Options: "270m", "1b", "4b", "12b", "27b"
# ───────────────────────────────────────────────────────────────────────────

MODEL_CFG = MODEL_CONFIGS[MODEL_SIZE]
KAGGLE_HANDLE = MODEL_CFG["kaggle_handle"]
CKPT_SUBDIR = MODEL_CFG["ckpt_subdir"]

print(f"Model size   : {MODEL_SIZE}")
print(f"Kaggle handle: {KAGGLE_HANDLE}")
print(f"Checkpoint   : {CKPT_SUBDIR}")

In [ ]:
ENCLAVE_EMAIL         = "test.enclave@gmail.com"
RESEARCHER_EMAIL      = "test.researcher@gmail.com"
BENCHMARK_OWNER_EMAIL = "test.benchmark.owner@gmail.com"

print(f"  Enclave    : {ENCLAVE_EMAIL}\n  Researcher : {RESEARCHER_EMAIL}\n  Benchmark  : {BENCHMARK_OWNER_EMAIL}")

## Step 0 — Log in as Model Owner

In [ ]:
model_owner = login_do()
print(f"  Model owner : {model_owner.email}")

In [ ]:
# # Optionally to clear state
# model_owner._manager.delete_syftbox()
# model_owner._manager.peer_manager.write_own_version()

### Launch the enclave

## Step 1 — Peer with the Enclave

In [ ]:
model_owner.add_peer(ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until you see their request appear, then approve.

In [ ]:
model_owner.sync()
model_owner.peers

In [ ]:
model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
model_owner.sync()
print("  Researcher peer approved")

### Step 1.2 — Attest enclave's identity

In [ ]:
# Wait for enclave to accept peer request
model_owner.attest_peer(ENCLAVE_EMAIL)

## Step 2 — Mount Gemma 3 Weights from Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

GEMMA_3_WEIGHTS_DIR = Path(f"~/.cache/kagglehub/models/google/gemma-3/flax").expanduser().absolute()
GEMMA_3_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
# copy drive weights to cache if not already present
!cp -R /content/drive/MyDrive/gemma-3-270m-it ~/.cache/kagglehub/models/google/gemma-3/flax



In [ ]:
weights_dir = Path(f"~/.cache/kagglehub/models/{KAGGLE_HANDLE}/{CKPT_SUBDIR}").expanduser().absolute()

## Step 3 — Build the private dataset

Model owner's private contribution is a directory containing:
- `gemma_inference.py` — the inference engine (model architecture + generate function)
- `{CKPT_SUBDIR}/` — the checkpoint weights directory
- `tokenizer.model` — the SentencePiece tokenizer

The **mock** (public) side is just a model card describing the model.

In [ ]:
# Build Model owner's private dataset directory: inference code + weights + tokenizer
INFERENCE_MODULE = Path("gemma_inference_restrict.py").resolve()
assert INFERENCE_MODULE.exists(), f"Missing {INFERENCE_MODULE}"


def create_model_private_dir() -> Path:
    """Bundle inference code + weights into a single directory."""
    tmp = Path(tempfile.mkdtemp()) / f"gemma3-private-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)

    # Copy inference module (uploaded under its canonical name)
    shutil.copy2(INFERENCE_MODULE, tmp / "gemma_inference.py")

    # Copy tokenizer
    shutil.copy2(Path(weights_dir) / "tokenizer.model", tmp / "tokenizer.model")

    # Copy checkpoint directory
    ckpt_src = Path(weights_dir) / CKPT_SUBDIR
    shutil.copytree(ckpt_src, tmp / CKPT_SUBDIR)

    return tmp


def create_model_mock_file() -> Path:
    """Public model card — visible to the researcher."""
    tmp = Path(tempfile.mkdtemp()) / f"model-mock-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "model_card.txt"
    p.write_text("\n".join([
        f"Gemma 3 {MODEL_SIZE.upper()}-IT: A {MODEL_SIZE} parameter instruction-tuned language model.",
        f"{'=' * (len(MODEL_SIZE) + 12)}",
        "License: Gemma Terms of Use",
        "Intended use: Research and evaluation purposes",
        "",
        "Usage:",
        "  import gemma_inference as gemma",
        f'  model, tokenizer, params = gemma.setup_model("{MODEL_SIZE}", weights_dir)',
        '  response, stats = gemma.generate(model, params, tokenizer, "Your prompt here")',
        "",
    ]))
    return p

In [ ]:
model_private_dir = create_model_private_dir()
model_mock = create_model_mock_file()

print(f"Private dir contents:")
for item in sorted(model_private_dir.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  {item.relative_to(model_private_dir)}  ({size_mb:.1f} MB)")

## Step 4 — Upload gemma3_model

Mock = model card; private = weights + inference code; shared with the enclave so it can run inference.

In [ ]:
model_owner.create_dataset(
    name="gemma3_model",
    mock_path=model_mock,
    private_path=model_private_dir,
    summary=f"Gemma 3 {MODEL_SIZE.upper()}-IT — instruction-tuned language model for safety evaluation",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    sync=False,
)
model_owner.share_private_dataset("gemma3_model", ENCLAVE_EMAIL)
model_owner.sync()
print("  Model owner uploaded 'gemma3_model'")

## Step 5 — Have the enclave run `syft-restrict` on the engine

The engine's private region is marked in the file with `# syft-restrict: ...` comments, so `run()` needs no line
ranges. The job installs `syft-restrict` from PyPI; it needs no JAX — it analyses the source, it doesn't run it.
Results go to the Benchmark Owner, who is named with an empty dataset list: recipient of the verdict, no data access.

In [ ]:
# The exact JAX/Flax leaves the private region may call; everything else is rejected.
ALLOW_FUNCTIONS = [
    "jax.numpy.einsum", "jax.numpy.mean", "jax.numpy.square", "jax.numpy.arange",
    "jax.numpy.sin", "jax.numpy.cos", "jax.numpy.concatenate", "jax.numpy.tril",
    "jax.numpy.triu", "jax.numpy.ones", "jax.numpy.where", "jax.numpy.repeat",
    "jax.numpy.sqrt", "jax.numpy.transpose", "jax.numpy.array", "jax.numpy.float32",
    "jax.numpy.bool_", "jax.lax.rsqrt", "jax.nn.softmax", "jax.nn.gelu",
    "flax.linen.Module", "jax.lax", "jax.nn",
]
ALLOW_OPERATORS = ["arithmetic", "indexing", "comparison"]

RESTRICT_JOB_NAME = "restrict_engine_review"
RESTRICT_JOB_CODE = f'''
import json
import os

import syft_client as sc
import syft_restrict as restrict

files = sc.resolve_dataset_files_path("gemma3_model", owner_email="{model_owner.email}")
src_path = [p for p in files if p.name == "gemma_inference.py"][0]

os.makedirs("outputs", exist_ok=True)
result = restrict.run(
    src_path,
    allow_functions={ALLOW_FUNCTIONS},
    allow_operators={ALLOW_OPERATORS},
    out="outputs/gemma_inference.obfuscated.py",
    strict=False,
)

if not result.ok:
    for v in result.violations:
        print(f"  line {{v.line}} [{{v.code}}] {{v.message}}")
    raise SystemExit(1)

with open("outputs/gemma_inference.certificate.json", "w") as f:
    json.dump(result.certificate, f, indent=2)

print(f"RESTRICT PASSED — {{result.certificate['n_calls_checked']}} calls checked")
print(f"  policy id     : {{result.certificate['policy_id']}}")
print(f"  source sha256 : {{result.certificate['source_sha256']}}")
'''


def create_code_file(code: str) -> str:
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)

In [ ]:
model_owner.submit_python_job(
    ENCLAVE_EMAIL,
    create_code_file(RESTRICT_JOB_CODE),
    RESTRICT_JOB_NAME,
    datasets={
        model_owner.email: ["gemma3_model"],
        BENCHMARK_OWNER_EMAIL: [],  # no data — recipient of the verdict only
    },
    share_results_with_do=True,
    dependencies=["syft-restrict"],
)
model_owner.sync()
print(f"  Job '{RESTRICT_JOB_NAME}' submitted to the enclave ({ENCLAVE_EMAIL})")

### Step 5.1 — Approve it

Both Data Owners must approve before the enclave will run it. Re-sync until it appears, then approve.

In [ ]:
model_owner.sync()
restrict_job = next(j for j in model_owner.jobs if j.name == RESTRICT_JOB_NAME)
print(f"  Model owner sees '{RESTRICT_JOB_NAME}'  status={restrict_job.status}")

In [ ]:
restrict_job

In [ ]:
model_owner.approve_job(restrict_job)
model_owner.sync()
print("  Model owner approved")

## Step 6 — Wait for the Researcher to submit the job, then approve

The Researcher submits `safety_eval_job` to the enclave. Re-sync until it appears here, inspect it, then approve.

In [ ]:
JOB_NAME = "safety_eval_job"
model_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
print(f"  Model owner sees '{JOB_NAME}'  status={model_owner_job.status}")

In [ ]:
model_owner_job

In [ ]:
model_owner.approve_job(model_owner_job)
model_owner.sync()
print("  Model owner approved")